# Practical 5 - Compound Queries & Advanced Filtering

This practical builds **compound queries** (`bool` with `must`, `filter`,
`should`, `must_not`) and compares scoring context with filter context, using
the official `elasticsearch` Python client against Elasticsearch 9.x.

Covered requirements:

1. Active employees in Mumbai.
2. Engineering or Sales employees.
3. Salary strictly between 50000 and 100000.
4. Electronics products excluding unavailable ones.
5. Products matching multiple search conditions.
6. Products with multiple preferred keywords.
7. `should` with `minimum_should_match`.
8. Boosting a preferred category.
9. Nested queries (with `inner_hits`).
10. `must` vs `filter` behaviour, plus a five-condition compound query.

Each experiment uses the `run()` helper, which prints the **requirement,
query, options, expected output, actual output and explanation**, and asserts
exact counts where the result is deterministic.

The notebook is idempotent: it recreates the `employees` and `products`
indices and reloads the JSON datasets on every run.

## Setup

Uses the shared repository virtual environment (kernel **`es`**). Prerequisites:

- Elasticsearch running (Practical 1).
- `python generate_data.py` executed in this folder.
- Kernel `es` registered from the repo-root `.venv` (see `README.md`).

In [1]:
import json
import re
from pathlib import Path

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

ES_URL = "http://127.0.0.1:9200"
es = Elasticsearch(ES_URL)

info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])

HITS_SHOWN = 3
SUMMARY = []


def pretty(obj):
    return json.dumps(obj, indent=2, default=str)


def hits_of(response):
    return response["hits"]["hits"]


def total_hits(response):
    return response["hits"]["total"]["value"]


def ids_of(response):
    return [hit["_id"] for hit in hits_of(response)]


def tokens(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def recreate(index, mappings):
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
    es.indices.create(
        index=index,
        settings={"number_of_shards": 1, "number_of_replicas": 0},
        mappings=mappings,
    )
    print("created index:", index)


def bulk_load(index, documents, id_field):
    actions = [
        {"_index": index, "_id": str(document[id_field]), "_source": document}
        for document in documents
    ]
    indexed, errors = bulk(es, actions, refresh=True)
    suffix = f" ({len(errors)} errors)" if errors else ""
    print(f"indexed {indexed} documents into {index}{suffix}")


def label(hit, index):
    source = hit.get("_source", {}) or {}
    if index == "employees":
        parts = [hit["_id"], source.get("name"), source.get("department"),
                 source.get("city")]
        if "salary" in source:
            parts.append(f"salary {source['salary']}")
    else:
        parts = [hit["_id"], source.get("name"), source.get("category")]
        if "price" in source:
            parts.append(f"price {source['price']}")
    return " | ".join(str(part) for part in parts if part)


def print_scores(response, index, title):
    print(f"  {title}")
    print(f"    {'_id':<6}{'_score':>9}  document")
    for hit in hits_of(response):
        print(f"    {hit['_id']:<6}{hit['_score']:>9.4f}  {label(hit, index)}")


def run(number, title, requirement, query, expected, explanation, index,
        expected_hits=None, record=None, display=True, **search_params):
    response = es.search(index=index, query=query, **search_params)
    count = total_hits(response)

    print(f"{number} | {title}")
    print("  Requirement:")
    print("   ", requirement)
    print("  Query:")
    for line in pretty(query).splitlines():
        print("   ", line)
    if search_params:
        shown = {("from" if key == "from_" else key): value
                 for key, value in search_params.items()}
        print("  Options:")
        for line in pretty(shown).splitlines():
            print("   ", line)
    print("  Expected:")
    print("   ", expected)
    print(f"  Actual: {count} hit(s)")
    if display:
        for hit in hits_of(response)[:HITS_SHOWN]:
            print("    -", label(hit, index))
        if count > HITS_SHOWN:
            print(f"    ... {count - HITS_SHOWN} more")
    print("  Explanation:")
    print("   ", explanation)
    print("-" * 100)

    if expected_hits is not None and count != expected_hits:
        raise AssertionError(f"{number}: expected {expected_hits} hits, got {count}")
    if record:
        SUMMARY.append({"query": number, "requirement": requirement, "hits": count})
    return response

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node


### Data and mappings

`products` adds a **nested** `reviews` field, an ordinary object copy
`reviews_flat` (used to demonstrate the nested pitfall) and a `supplier`
object. `employees` keeps the Practical 2/4 schema.

In [2]:
HERE = Path.cwd()
employees_data = json.loads((HERE / "employees.json").read_text(encoding="utf-8"))
products_data = json.loads((HERE / "products.json").read_text(encoding="utf-8"))
print("employees loaded:", len(employees_data))
print("products loaded:", len(products_data))

employees_mapping = {
    "properties": {
        "employee_id": {"type": "integer"},
        "name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
        "department": {"type": "keyword"},
        "designation": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
        "salary": {"type": "integer"},
        "city": {"type": "keyword"},
        "joining_date": {"type": "date"},
        "active": {"type": "boolean"},
        "email": {"type": "keyword"},
        "bonus": {"type": "float"},
    }
}

review_properties = {
    "author": {"type": "keyword"},
    "rating": {"type": "integer"},
    "comment": {"type": "text"},
    "date": {"type": "date"},
}

products_mapping = {
    "properties": {
        "product_id": {"type": "keyword"},
        "name": {"type": "text",
                 "fields": {"keyword": {"type": "keyword", "ignore_above": 256}}},
        "description": {"type": "text"},
        "brand": {"type": "keyword"},
        "category": {"type": "keyword"},
        "subcategory": {"type": "keyword"},
        "price": {"type": "float"},
        "stock": {"type": "integer"},
        "rating": {"type": "float"},
        "tags": {"type": "keyword"},
        "available": {"type": "boolean"},
        "created_at": {"type": "date"},
        "sku": {"type": "keyword"},
        "warranty_months": {"type": "integer"},
        "supplier": {
            "properties": {
                "name": {"type": "keyword"},
                "country": {"type": "keyword"},
                "lead_time_days": {"type": "integer"},
            }
        },
        "reviews": {"type": "nested", "properties": review_properties},
        "reviews_flat": {"properties": review_properties},
    }
}

recreate("employees", employees_mapping)
bulk_load("employees", employees_data, "employee_id")
recreate("products", products_mapping)
bulk_load("products", products_data, "product_id")

print()
print("employees count:", es.count(index="employees")["count"])
print("products count:", es.count(index="products")["count"])


def emp_where(predicate):
    return sum(1 for employee in employees_data if predicate(employee))


def prod_where(predicate):
    return sum(1 for product in products_data if predicate(product))

employees loaded: 120
products loaded: 60


created index: employees
indexed 120 documents into employees


created index: products


indexed 60 documents into products

employees count: 120
products count: 60


## 1. Active employees in Mumbai

Both conditions are exact, so `filter` context is the right choice: no scoring,
and the results can be cached.

In [3]:
run(
    "Q1", "active employees in Mumbai",
    "Find employees who are active and based in Mumbai.",
    {"bool": {
        "filter": [
            {"term": {"active": True}},
            {"term": {"city": "Mumbai"}},
        ]
    }},
    f"{emp_where(lambda e: e['active'] and e['city'] == 'Mumbai')} employees",
    "bool.filter ANDs the two exact terms and runs in filter context, so the "
    "query is fast and cacheable and contributes no relevance score.",
    "employees",
    expected_hits=emp_where(lambda e: e["active"] and e["city"] == "Mumbai"),
    record=True,
)

Q1 | active employees in Mumbai
  Requirement:
    Find employees who are active and based in Mumbai.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "active": true
            }
          },
          {
            "term": {
              "city": "Mumbai"
            }
          }
        ]
      }
    }
  Expected:
    10 employees
  Actual: 10 hit(s)
    - 118 | Vikram Rao | Finance | Mumbai | salary 111000
    - 132 | Priya Desai | Finance | Mumbai | salary 38500
    - 134 | Gaurav Agarwal | Support | Mumbai | salary 122500
    ... 7 more
  Explanation:
    bool.filter ANDs the two exact terms and runs in filter context, so the query is fast and cacheable and contributes no relevance score.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 11, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 10, 'relation': 'eq'}, 'max_score': 0.0, 'hits': [{'_index': 'employees', '_id': '118', '_score': 0.0, '_source': {'employee_id': 118, 'name': 'Vikram Rao', 'department': 'Finance', 'designation': 'Finance Manager', 'salary': 111000, 'city': 'Mumbai', 'joining_date': '2024-09-19', 'active': True, 'email': 'vikram.rao118@example.com'}}, {'_index': 'employees', '_id': '132', '_score': 0.0, '_source': {'employee_id': 132, 'name': 'Priya Desai', 'department': 'Finance', 'designation': 'Financial Analyst', 'salary': 38500, 'city': 'Mumbai', 'joining_date': '2024-07-20', 'active': True}}, {'_index': 'employees', '_id': '134', '_score': 0.0, '_source': {'employee_id': 134, 'name': 'Gaurav Agarwal', 'department': 'Support', 'designation': 'Customer Success Manager', 'salary': 122500, 'city': 'Mumbai', 'joining_date': '2024-12-11', 'active': True, '

## 2. Employees in Engineering or Sales

Two equivalent formulations: `terms` (compact) and `bool.should` with
`minimum_should_match: 1` (explicit).

In [4]:
run(
    "Q2a", "terms filter",
    "Find employees in Engineering or Sales.",
    {"terms": {"department": ["Engineering", "Sales"]}},
    f"{emp_where(lambda e: e['department'] in ('Engineering', 'Sales'))} employees",
    "terms matches any of the listed exact values in a single clause.",
    "employees",
    expected_hits=emp_where(lambda e: e["department"] in ("Engineering", "Sales")),
    record=True,
)

run(
    "Q2b", "bool should with minimum_should_match",
    "Express the same OR with two should clauses.",
    {"bool": {
        "should": [
            {"term": {"department": "Engineering"}},
            {"term": {"department": "Sales"}},
        ],
        "minimum_should_match": 1,
    }},
    f"{emp_where(lambda e: e['department'] in ('Engineering', 'Sales'))} employees",
    "should clauses are optional by default; minimum_should_match: 1 turns the "
    "group into an OR. Without it, a bool query with only should clauses still "
    "defaults to 1, but being explicit is safer.",
    "employees",
    expected_hits=emp_where(lambda e: e["department"] in ("Engineering", "Sales")),
)

Q2a | terms filter
  Requirement:
    Find employees in Engineering or Sales.
  Query:
    {
      "terms": {
        "department": [
          "Engineering",
          "Sales"
        ]
      }
    }
  Expected:
    35 employees
  Actual: 35 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 102 | Gaurav Kapoor | Sales | Kolkata | salary 40500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    ... 32 more
  Explanation:
    terms matches any of the listed exact values in a single clause.
----------------------------------------------------------------------------------------------------


Q2b | bool should with minimum_should_match
  Requirement:
    Express the same OR with two should clauses.
  Query:
    {
      "bool": {
        "should": [
          {
            "term": {
              "department": "Engineering"
            }
          },
          {
            "term": {
              "department": "Sales"
            }
          }
        ],
        "minimum_should_match": 1
      }
    }
  Expected:
    35 employees
  Actual: 35 hit(s)
    - 102 | Gaurav Kapoor | Sales | Kolkata | salary 40500
    - 109 | Shreya Reddy | Sales | Bengaluru | salary 122000
    - 116 | Suresh Gupta | Sales | Kolkata | salary 128500
    ... 32 more
  Explanation:
    should clauses are optional by default; minimum_should_match: 1 turns the group into an OR. Without it, a bool query with only should clauses still defaults to 1, but being explicit is safer.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 19, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 35, 'relation': 'eq'}, 'max_score': 1.9335895, 'hits': [{'_index': 'employees', '_id': '102', '_score': 1.9335895, '_source': {'employee_id': 102, 'name': 'Gaurav Kapoor', 'department': 'Sales', 'designation': 'Sales Executive', 'salary': 40500, 'city': 'Kolkata', 'joining_date': '2018-05-11', 'active': True, 'email': 'gaurav.kapoor102@example.com'}}, {'_index': 'employees', '_id': '109', '_score': 1.9335895, '_source': {'employee_id': 109, 'name': 'Shreya Reddy', 'department': 'Sales', 'designation': 'Account Manager', 'salary': 122000, 'city': 'Bengaluru', 'joining_date': '2023-03-09', 'active': True}}, {'_index': 'employees', '_id': '116', '_score': 1.9335895, '_source': {'employee_id': 116, 'name': 'Suresh Gupta', 'department': 'Sales', 'designation': 'Sales Manager', 'salary': 128500, 'city': 'Kolkata', 'joining_date': '2019-10-10', 'a

## 3. Salary more than 50000 but less than 100000

`gt`/`lt` are exclusive. The `gte`/`lte` variant is shown for contrast; both
return the same count here because no employee earns exactly 50000 or 100000.

In [5]:
run(
    "Q3", "exclusive salary range",
    "Find employees earning more than 50000 but less than 100000.",
    {"range": {"salary": {"gt": 50000, "lt": 100000}}},
    f"{emp_where(lambda e: 50000 < e['salary'] < 100000)} employees",
    "gt and lt exclude the boundary values. range runs in query context here, "
    "but wrapping it in bool.filter would be cheaper because salary is only a "
    "business constraint.",
    "employees",
    expected_hits=emp_where(lambda e: 50000 < e["salary"] < 100000),
    record=True,
)

run(
    "Q3b", "inclusive salary range for comparison",
    "Same band but keeping the boundaries.",
    {"range": {"salary": {"gte": 50000, "lte": 100000}}},
    f"{emp_where(lambda e: 50000 <= e['salary'] <= 100000)} employees",
    "gte/lte include boundary salaries. The count matches the exclusive query "
    "because no employee falls exactly on 50000 or 100000.",
    "employees",
    expected_hits=emp_where(lambda e: 50000 <= e["salary"] <= 100000),
)

Q3 | exclusive salary range
  Requirement:
    Find employees earning more than 50000 but less than 100000.
  Query:
    {
      "range": {
        "salary": {
          "gt": 50000,
          "lt": 100000
        }
      }
    }
  Expected:
    48 employees
  Actual: 48 hit(s)
    - 104 | Vishal Reddy | Finance | Hyderabad | salary 62500
    - 105 | Manish Gupta | Marketing | Hyderabad | salary 58500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    ... 45 more
  Explanation:
    gt and lt exclude the boundary values. range runs in query context here, but wrapping it in bool.filter would be cheaper because salary is only a business constraint.
----------------------------------------------------------------------------------------------------


Q3b | inclusive salary range for comparison
  Requirement:
    Same band but keeping the boundaries.
  Query:
    {
      "range": {
        "salary": {
          "gte": 50000,
          "lte": 100000
        }
      }
    }
  Expected:
    48 employees
  Actual: 48 hit(s)
    - 104 | Vishal Reddy | Finance | Hyderabad | salary 62500
    - 105 | Manish Gupta | Marketing | Hyderabad | salary 58500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    ... 45 more
  Explanation:
    gte/lte include boundary salaries. The count matches the exclusive query because no employee falls exactly on 50000 or 100000.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 17, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 48, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'employees', '_id': '104', '_score': 1.0, '_source': {'employee_id': 104, 'name': 'Vishal Reddy', 'department': 'Finance', 'designation': 'Accountant', 'salary': 62500, 'city': 'Hyderabad', 'joining_date': '2021-02-12', 'active': True, 'bonus': 5000.0}}, {'_index': 'employees', '_id': '105', '_score': 1.0, '_source': {'employee_id': 105, 'name': 'Manish Gupta', 'department': 'Marketing', 'designation': 'Marketing Executive', 'salary': 58500, 'city': 'Hyderabad', 'joining_date': '2024-10-08', 'active': True, 'email': 'manish.gupta105@example.com', 'bonus': 5000.0}}, {'_index': 'employees', '_id': '108', '_score': 1.0, '_source': {'employee_id': 108, 'name': 'Manish Mehta', 'department': 'Engineering', 'designation': 'Senior Software Engineer', 'salary': 99500, 'city': 'Hyderabad', 

## 4. Electronics excluding unavailable products

Two equivalent patterns: require `available: true`, or exclude
`available: false` with `must_not`.

In [6]:
run(
    "Q4", "category filter plus availability filter",
    "Find Electronics products that are available.",
    {"bool": {
        "filter": [
            {"term": {"category": "Electronics"}},
            {"term": {"available": True}},
        ]
    }},
    f"{prod_where(lambda p: p['category'] == 'Electronics' and p['available'])} products",
    "Both clauses are exact and run in filter context; only the available "
    "Electronics products are returned.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] == "Electronics" and p["available"]
    ),
    record=True,
)

run(
    "Q4b", "same result via must_not",
    "Exclude unavailable Electronics with must_not.",
    {"bool": {
        "must": [{"term": {"category": "Electronics"}}],
        "must_not": [{"term": {"available": False}}],
    }},
    f"{prod_where(lambda p: p['category'] == 'Electronics' and p['available'])} products",
    "must_not removes documents matching available: false. This is logically "
    "the same as requiring available: true, but must_not also excludes "
    "documents where the field is missing.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] == "Electronics" and p["available"]
    ),
)

Q4 | category filter plus availability filter
  Requirement:
    Find Electronics products that are available.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "category": "Electronics"
            }
          },
          {
            "term": {
              "available": true
            }
          }
        ]
      }
    }
  Expected:
    9 products
  Actual: 9 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P004 | Ultra Slim Laptop 14 inch | Electronics | price 78999.0
    ... 6 more
  Explanation:
    Both clauses are exact and run in filter context; only the available Electronics products are returned.
----------------------------------------------------------------------------------------------------


Q4b | same result via must_not
  Requirement:
    Exclude unavailable Electronics with must_not.
  Query:
    {
      "bool": {
        "must": [
          {
            "term": {
              "category": "Electronics"
            }
          }
        ],
        "must_not": [
          {
            "term": {
              "available": false
            }
          }
        ]
      }
    }
  Expected:
    9 products
  Actual: 9 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P004 | Ultra Slim Laptop 14 inch | Electronics | price 78999.0
    ... 6 more
  Explanation:
    must_not removes documents matching available: false. This is logically the same as requiring available: true, but must_not also excludes documents where the field is missing.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 17, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 9, 'relation': 'eq'}, 'max_score': 1.7594986, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 1.7594986, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 14, 'rating': 3.2, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': True, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12, 'supplier': {'name': 'AcmeSupply', 'country': 'IN', 'lead_time_days': 7}, 'reviews': [{'author': 'alice', 'rating': 5, 'comment': 'excellent quality and sound', 'date': '2024-03-12'}, {'author': 'bob', 'rating': 4, 'comment': 'good value for 

## 5. Products matching multiple search conditions

A realistic catalog query: full-text on the name plus exact brand, price and
date constraints. Scoring clauses stay in `must`; constraints go in `filter`.

In [7]:
run(
    "Q5", "multi-condition product search",
    "Find Acme wireless products priced 100-10000 created since 2023.",
    {"bool": {
        "must": [{"match": {"name": "wireless"}}],
        "filter": [
            {"term": {"brand": "Acme"}},
            {"range": {"price": {"gte": 100, "lte": 10000}}},
            {"range": {"created_at": {"gte": "2023-01-01",
                                      "lte": "2025-12-31"}}},
        ]
    }},
    f"{prod_where(lambda p: 'wireless' in tokens(p['name']) and p['brand'] == 'Acme' and 100 <= p['price'] <= 10000)} products",
    "The match clause drives relevance; brand, price and date are exact filters. "
    "Because the filters are in filter context they do not dilute the score.",
    "products",
    expected_hits=prod_where(
        lambda p: "wireless" in tokens(p["name"])
        and p["brand"] == "Acme"
        and 100 <= p["price"] <= 10000
    ),
    record=True,
)

Q5 | multi-condition product search
  Requirement:
    Find Acme wireless products priced 100-10000 created since 2023.
  Query:
    {
      "bool": {
        "must": [
          {
            "match": {
              "name": "wireless"
            }
          }
        ],
        "filter": [
          {
            "term": {
              "brand": "Acme"
            }
          },
          {
            "range": {
              "price": {
                "gte": 100,
                "lte": 10000
              }
            }
          },
          {
            "range": {
              "created_at": {
                "gte": "2023-01-01",
                "lte": "2025-12-31"
              }
            }
          }
        ]
      }
    }
  Expected:
    2 products
  Actual: 2 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P007 | Wireless Noise-Cancelling Earbuds | Electronics | price 7999.0
  Explanation:
    The match clause drives relevan

ObjectApiResponse({'took': 19, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 2.580325, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 2.580325, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 14, 'rating': 3.2, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': True, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12, 'supplier': {'name': 'AcmeSupply', 'country': 'IN', 'lead_time_days': 7}, 'reviews': [{'author': 'alice', 'rating': 5, 'comment': 'excellent quality and sound', 'date': '2024-03-12'}, {'author': 'bob', 'rating': 4, 'comment': 'good value for th

## 6. Multiple preferred keywords

Put each keyword in its own `should` clause. With only `should` clauses and the
default `minimum_should_match: 1`, a document matches when **any** keyword is
present, and the scores combine across keywords (BM25 sums the matching
clauses).

In [8]:
KEYWORDS = ["wireless", "audio", "noise", "cancellation"]

keyword_query = {"bool": {"should": [
    {"match": {"description": keyword}} for keyword in KEYWORDS
]}}

run(
    "Q6", "preferred keywords via should",
    "Find products describing any of: wireless, audio, noise, cancellation.",
    keyword_query,
    f"{prod_where(lambda p: any(k in tokens(p['description']) for k in KEYWORDS))} products",
    "Each should clause is optional; matching more keywords increases the "
    "score, which naturally ranks documents that satisfy several preferences "
    "above documents that satisfy one.",
    "products",
    expected_hits=prod_where(
        lambda p: any(keyword in tokens(p["description"]) for keyword in KEYWORDS)
    ),
    record=True,
)

Q6 | preferred keywords via should
  Requirement:
    Find products describing any of: wireless, audio, noise, cancellation.
  Query:
    {
      "bool": {
        "should": [
          {
            "match": {
              "description": "wireless"
            }
          },
          {
            "match": {
              "description": "audio"
            }
          },
          {
            "match": {
              "description": "noise"
            }
          },
          {
            "match": {
              "description": "cancellation"
            }
          }
        ]
      }
    }
  Expected:
    4 products
  Actual: 4 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P007 | Wireless Noise-Cancelling Earbuds | Electronics | price 7999.0
    - P006 | Wireless Charging Pad | Electronics | price 1499.0
    ... 1 more
  Explanation:
    Each should clause is optional; matching more keywords increases the score, which naturally rank

ObjectApiResponse({'took': 7, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 4, 'relation': 'eq'}, 'max_score': 9.486853, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 9.486853, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 14, 'rating': 3.2, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': True, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12, 'supplier': {'name': 'AcmeSupply', 'country': 'IN', 'lead_time_days': 7}, 'reviews': [{'author': 'alice', 'rating': 5, 'comment': 'excellent quality and sound', 'date': '2024-03-12'}, {'author': 'bob', 'rating': 4, 'comment': 'good value for the

## 7. `should` with `minimum_should_match`

Raising `minimum_should_match` turns preferences into requirements. The table
shows how the hit count drops as the threshold increases.

With only `should` clauses, Elasticsearch defaults `minimum_should_match` to 1
when there is no `must` or `filter` clause; specifying it explicitly makes the
intent clear.

In [9]:
for threshold in range(1, len(KEYWORDS) + 1):
    query = {"bool": {
        "should": [{"match": {"description": keyword}} for keyword in KEYWORDS],
        "minimum_should_match": threshold,
    }}
    response = es.search(index="products", query=query)
    expected = prod_where(
        lambda p: sum(1 for k in KEYWORDS if k in tokens(p["description"])) >= threshold
    )
    assert total_hits(response) == expected, (threshold, total_hits(response), expected)
    print(f"minimum_should_match={threshold}: {total_hits(response)} products")

run(
    "Q7", "should with minimum_should_match: 2",
    "Require at least two of the four preferred keywords.",
    {"bool": {
        "should": [{"match": {"description": keyword}} for keyword in KEYWORDS],
        "minimum_should_match": 2,
    }},
    f"{prod_where(lambda p: sum(1 for k in KEYWORDS if k in tokens(p['description'])) >= 2)} products",
    "minimum_should_match: 2 keeps only documents matching at least two should "
    "clauses. Raising it to 3 gives the same set here because the matching "
    "products satisfy three keywords; 4 would return nothing.",
    "products",
    expected_hits=prod_where(
        lambda p: sum(1 for k in KEYWORDS if k in tokens(p["description"])) >= 2
    ),
    record=True,
)

minimum_should_match=1: 4 products
minimum_should_match=2: 2 products
minimum_should_match=3: 2 products


minimum_should_match=4: 0 products
Q7 | should with minimum_should_match: 2
  Requirement:
    Require at least two of the four preferred keywords.
  Query:
    {
      "bool": {
        "should": [
          {
            "match": {
              "description": "wireless"
            }
          },
          {
            "match": {
              "description": "audio"
            }
          },
          {
            "match": {
              "description": "noise"
            }
          },
          {
            "match": {
              "description": "cancellation"
            }
          }
        ],
        "minimum_should_match": 2
      }
    }
  Expected:
    2 products
  Actual: 2 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P007 | Wireless Noise-Cancelling Earbuds | Electronics | price 7999.0
  Explanation:
    minimum_should_match: 2 keeps only documents matching at least two should clauses. Raising it to 3 gives the same set

ObjectApiResponse({'took': 11, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 9.486853, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 9.486853, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 14, 'rating': 3.2, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': True, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12, 'supplier': {'name': 'AcmeSupply', 'country': 'IN', 'lead_time_days': 7}, 'reviews': [{'author': 'alice', 'rating': 5, 'comment': 'excellent quality and sound', 'date': '2024-03-12'}, {'author': 'bob', 'rating': 4, 'comment': 'good value for th

## 8. Boost a preferred category

`must` matches `set`; a `should` clause adds a large boost when the product is
in Home & Kitchen. The boosted products were only mid-ranked before, so the
should clause clearly reorders the results without filtering any matches out.

In [10]:
plain = run(
    "Q8a", "relevance without a boost",
    "Find products whose name contains set.",
    {"match": {"name": "set"}},
    "all products containing set, ranked by BM25 only",
    "The baseline ranking depends only on the analyzed name match.",
    "products", display=False, size=5,
)

boosted = run(
    "Q8b", "relevance with a Sports category boost",
    "Prefer Home & Kitchen products in the set results.",
    {"bool": {
        "must": [{"match": {"name": "set"}}],
        "should": [{"term": {"category": {"value": "Home & Kitchen", "boost": 5}}}],
    }},
    "the same products, with the Home & Kitchen ones ranked first",
    "The should clause adds a boosted term score to matching documents. It is "
    "not required (minimum_should_match defaults to 0 when must is present), so "
    "it reorders results instead of filtering them.",
    "products", display=False, size=5, record=True,
)

print_scores(plain, "products", "without boost")
print_scores(boosted, "products", "with Home & Kitchen boost 5")
assert hits_of(boosted)[0]["_source"]["category"] == "Home & Kitchen"
assert hits_of(plain)[0]["_source"]["category"] != "Home & Kitchen"
print("boost reordered the results and ranks a Home & Kitchen product first")

Q8a | relevance without a boost
  Requirement:
    Find products whose name contains set.
  Query:
    {
      "match": {
        "name": "set"
      }
    }
  Options:
    {
      "size": 5
    }
  Expected:
    all products containing set, ranked by BM25 only
  Actual: 5 hit(s)
  Explanation:
    The baseline ranking depends only on the analyzed name match.
----------------------------------------------------------------------------------------------------
Q8b | relevance with a Sports category boost
  Requirement:
    Prefer Home & Kitchen products in the set results.
  Query:
    {
      "bool": {
        "must": [
          {
            "match": {
              "name": "set"
            }
          }
        ],
        "should": [
          {
            "term": {
              "category": {
                "value": "Home & Kitchen",
                "boost": 5
              }
            }
          }
        ]
      }
    }
  Options:
    {
      "size": 5
    }
  Expected:
    

## 9. Nested query

`reviews` is mapped as `nested`, so each review is indexed independently.
The same data is also stored in `reviews_flat` as a normal object array.

Requirement: products with a review by **alice** and a rating of **at least 4**.
- The flat query matches any product having alice *somewhere* and a rating >= 4
  *somewhere*, even when they come from different reviews.
- The nested query requires both conditions inside the **same** review.

Product P002 has alice/2 and bob/5, so it is a false positive for the flat
query and correctly excluded by the nested query.

In [11]:
flat = run(
    "Q9a", "object array (wrong correlation)",
    "Find products with a review by alice and a rating >= 4 using the flat field.",
    {"bool": {
        "must": [
            {"term": {"reviews_flat.author": "alice"}},
            {"range": {"reviews_flat.rating": {"gte": 4}}},
        ]
    }},
    f"{prod_where(lambda p: 'alice' in [r['author'] for r in p['reviews_flat']] and any(r['rating'] >= 4 for r in p['reviews_flat']))} products (includes P002)",
    "Object arrays are flattened, so the two conditions can be satisfied by "
    "different array elements. P002 (alice/2, bob/5) incorrectly matches.",
    "products", display=False,
    expected_hits=prod_where(
        lambda p: "alice" in [r["author"] for r in p["reviews_flat"]]
        and any(r["rating"] >= 4 for r in p["reviews_flat"])
    ),
)
assert "P002" in ids_of(flat)

nested = run(
    "Q9b", "nested query (correct correlation)",
    "Find products with a single review by alice rated at least 4.",
    {"nested": {
        "path": "reviews",
        "query": {"bool": {
            "must": [
                {"term": {"reviews.author": "alice"}},
                {"range": {"reviews.rating": {"gte": 4}}},
            ]
        }},
        "inner_hits": {"size": 3},
    }},
    f"{prod_where(lambda p: any(r['author'] == 'alice' and r['rating'] >= 4 for r in p['reviews']))} products (excludes P002)",
    "Each nested review is a separate hidden document, so both conditions must "
    "hold within one review. inner_hits returns the reviews that matched.",
    "products", display=False, record=True,
    expected_hits=prod_where(
        lambda p: any(r["author"] == "alice" and r["rating"] >= 4 for r in p["reviews"])
    ),
)
assert "P002" not in ids_of(nested)

for hit in hits_of(nested)[:3]:
    print(f"  {hit['_id']} {hit['_source']['name']}")
    for inner in hit["inner_hits"]["reviews"]["hits"]["hits"]:
        print("     matched review:", inner["_source"])
assert all(
    hit.get("inner_hits", {}).get("reviews", {}).get("hits", {}).get("hits")
    for hit in hits_of(nested)
)
print("nested correlation verified and inner_hits returned")

Q9a | object array (wrong correlation)
  Requirement:
    Find products with a review by alice and a rating >= 4 using the flat field.
  Query:
    {
      "bool": {
        "must": [
          {
            "term": {
              "reviews_flat.author": "alice"
            }
          },
          {
            "range": {
              "reviews_flat.rating": {
                "gte": 4
              }
            }
          }
        ]
      }
    }
  Expected:
    21 products (includes P002)
  Actual: 21 hit(s)
  Explanation:
    Object arrays are flattened, so the two conditions can be satisfied by different array elements. P002 (alice/2, bob/5) incorrectly matches.
----------------------------------------------------------------------------------------------------
Q9b | nested query (correct correlation)
  Requirement:
    Find products with a single review by alice rated at least 4.
  Query:
    {
      "nested": {
        "path": "reviews",
        "query": {
          "bool": {


## 10. `must` vs `filter` behaviour

The same five conditions are run once with the scoring clauses in `must` and
once with everything in `filter`:

1. Full-text: `match` on `name` (wireless).
2. Exact: `term` on `category` (Electronics).
3. Numeric range: `price` between 100 and 10000.
4. Date range: `created_at` in 2023-2025.
5. Exclusion: `must_not` unavailable products.

`must` computes a relevance score; `filter` returns score 0, can be cached, and
is cheaper. `_explain` confirms the difference for a single document.

In [12]:
scoring_clauses = [
    {"match": {"name": "wireless"}},
    {"term": {"category": "Electronics"}},
    {"range": {"price": {"gte": 100, "lte": 10000}}},
    {"range": {"created_at": {"gte": "2023-01-01", "lte": "2025-12-31"}}},
]

must_query = {"bool": {
    "must": scoring_clauses,
    "must_not": [{"term": {"available": False}}],
}}
filter_query = {"bool": {
    "filter": scoring_clauses,
    "must_not": [{"term": {"available": False}}],
}}

expected = prod_where(
    lambda p: "wireless" in tokens(p["name"])
    and p["category"] == "Electronics"
    and 100 <= p["price"] <= 10000
    and "2023-01-01" <= p["created_at"] <= "2025-12-31"
    and p["available"]
)

must_response = run(
    "Q10a", "five conditions in must (scored)",
    "Run the compound catalog query in scoring context.",
    must_query,
    f"{expected} products, each with a positive _score",
    "Clauses in must participate in scoring, so the final _score reflects how "
    "well each product matches the full-text clause.",
    "products", expected_hits=expected, display=False,
    record=True,
)

filter_response = run(
    "Q10b", "five conditions in filter (unscored)",
    "Run the identical compound catalog query in filter context.",
    filter_query,
    f"{expected} products, every _score equal to 0",
    "filter clauses are yes/no constraints. They skip scoring, return _score 0 "
    "and are cacheable, which is why business filters belong here.",
    "products", expected_hits=expected, display=False,
)

print_scores(must_response, "products", "must context")
print_scores(filter_response, "products", "filter context")
assert all(hit["_score"] > 0 for hit in hits_of(must_response))
assert all(hit["_score"] == 0 for hit in hits_of(filter_response))

Q10a | five conditions in must (scored)
  Requirement:
    Run the compound catalog query in scoring context.
  Query:
    {
      "bool": {
        "must": [
          {
            "match": {
              "name": "wireless"
            }
          },
          {
            "term": {
              "category": "Electronics"
            }
          },
          {
            "range": {
              "price": {
                "gte": 100,
                "lte": 10000
              }
            }
          },
          {
            "range": {
              "created_at": {
                "gte": "2023-01-01",
                "lte": "2025-12-31"
              }
            }
          }
        ],
        "must_not": [
          {
            "term": {
              "available": false
            }
          }
        ]
      }
    }
  Expected:
    3 products, each with a positive _score
  Actual: 3 hit(s)
  Explanation:
    Clauses in must participate in scoring, so the final _score r

### `_explain` output

`_explain` shows how Elasticsearch arrives at a document's score. In filter
context the explanation is a constant score of 0, while in `must` context it
walks through BM25 for the matching clause.

In [13]:
doc_id = ids_of(must_response)[0]

must_explain = es.explain(index="products", id=doc_id, query=must_query)
filter_explain = es.explain(index="products", id=doc_id, query=filter_query)

must_value = must_explain["explanation"]["value"]
filter_value = filter_explain["explanation"]["value"]
print(f"{doc_id} in must context  : score {must_value:.4f} -> "
      f"{must_explain['explanation']['description']}")
print(f"{doc_id} in filter context: score {filter_value:.4f} -> "
      f"{filter_explain['explanation']['description']}")

assert must_value > 0
assert filter_value == 0
print("explain confirms scored vs constant (0) evaluation")

P006 in must context  : score 6.6805 -> sum of:
P006 in filter context: score 0.0000 -> sum of:
explain confirms scored vs constant (0) evaluation


## Summary

Every requirement is recorded below with its total match count.

In [14]:
print(f"{'Query':<7}{'Matches':>8}  Requirement")
print("-" * 105)
for row in SUMMARY:
    print(f"{row['query']:<7}{row['hits']:>8}  {row['requirement'][:80]}")
print()
print("documented requirements:", len(SUMMARY))
assert len(SUMMARY) >= 10

Query   Matches  Requirement
---------------------------------------------------------------------------------------------------------
Q1           10  Find employees who are active and based in Mumbai.
Q2a          35  Find employees in Engineering or Sales.
Q3           48  Find employees earning more than 50000 but less than 100000.
Q4            9  Find Electronics products that are available.
Q5            2  Find Acme wireless products priced 100-10000 created since 2023.
Q6            4  Find products describing any of: wireless, audio, noise, cancellation.
Q7            2  Require at least two of the four preferred keywords.
Q8b           5  Prefer Home & Kitchen products in the set results.
Q9b           9  Find products with a single review by alice rated at least 4.
Q10a          3  Run the compound catalog query in scoring context.

documented requirements: 10


## Wrap-up

- `bool.filter` for exact/business constraints: no scoring, cacheable, cheap.
  `bool.must` only for clauses whose relevance should influence ranking.
- `must`/`filter` AND their clauses; `should` ORs them; `must_not` excludes.
  `minimum_should_match` converts optional preferences into requirements.
- `boost` on a `should` clause nudges ranking without excluding other matches.
- `nested` keeps array-of-object conditions inside one element; plain object
  arrays flatten and produce false positives.
- Use `inner_hits` to return the nested elements that matched.
- Validate compound queries with exact assertions; add `_explain` when scores
  look surprising.